# 09 — Development explanation pilot

**Objective.** Use 2005–2008 only to benchmark grouped attribution runtime/error, background sensitivity, Gower-kNN diagnostics, and cross-fitted explanation-loss meta-scores.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Permitted blocks: 2005–2008 only. Any budget change after this pilot must be uniform and frozen before final-test execution.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("09", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import json, time, joblib, yaml
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from cruxvc.esrc import case_construct_fragility_loss, case_reference_panel_loss
from cruxvc.explanations import GowerKNNDonorSampler, active_feature_groups, approximation_repeat_diagnostics, explain_dataset, load_feature_groups, proportional_stratified_sample
from cruxvc.faithfulness import faithfulness_audit, normalized_faithfulness_loss
from cruxvc.io import read_table, write_json, write_table
from cruxvc.preprocessing import build_preprocessor, infer_feature_types
from cruxvc.selective import standard_gate_scores

features = read_table(P.processed / "features_strict.parquet")
cohort = read_table(P.processed / "cohort_labels.parquet")
splits = read_table(P.protocol / "split_ids.parquet")
calibrated = read_table(P.models / "calibration_registry.parquet")
background_ids = pd.read_csv(P.protocol / "explanation_background_ids.csv")
seed_table = pd.read_csv(P.protocol / "seed_registry.csv")
frame = features.merge(cohort, on=["case_id", "company_permalink", "t0"]).merge(splits[["case_id", "time_block"]], on="case_id")
development = frame[frame["time_block"].astype(str).eq("development")].copy()
calibration_2008 = frame[frame["time_block"].astype(str).eq("probability_calibration")].copy()
feature_columns = [c for c in features.columns if c not in {"case_id", "company_permalink", "t0"}]
groups = active_feature_groups(load_feature_groups(P.config / "feature_groups.yaml"), feature_columns)
statistical = yaml.safe_load((P.config / "statistical_analysis.yaml").read_text(encoding="utf-8"))

In [ ]:
pilot_n = min(len(calibration_2008), max(60, int(PROFILE["local_audit_n"])))
pilot_cases = proportional_stratified_sample(
    calibration_2008,
    n=pilot_n,
    strata=["landmark_round_type"],
    seed=int(CFG["execution"]["random_seed"]),
)
active_background_ids = sorted(background_ids["background_id"].unique())[: int(PROFILE["background_sets"])]
backgrounds = {}
for background_id in active_background_ids:
    ids = (
        background_ids[background_ids["background_id"].eq(background_id)]
        .sort_values("order")["case_id"]
        .head(int(PROFILE["background_n"]))
    )
    backgrounds[background_id] = development[development["case_id"].isin(ids)].copy()
approximation_seeds = (
    seed_table[seed_table["purpose"].eq("approximation")]
    .sort_values("index")["seed"]
    .head(int(PROFILE["approximation_seeds"]))
    .astype(int)
    .tolist()
)
jobs = calibrated[calibrated["analysis_role"].eq("matched_reference")].copy()

In [ ]:
attribution_parts = []
runtime_rows = []
prediction_parts = []
for job in jobs.itertuples(index=False):
    model = joblib.load(job.calibrated_model_path)
    probability = model.predict_proba(calibration_2008)[:, 1]
    prediction_parts.append(pd.DataFrame({
        "case_id": calibration_2008["case_id"], "outcome": job.outcome, "family": job.family,
        "config_id": job.config_id, "probability": probability,
    }))
    for background_id, background in backgrounds.items():
        for approximation_seed in approximation_seeds:
            started = time.perf_counter()
            attribution_parts.append(explain_dataset(
                model, pilot_cases, background, groups, feature_columns,
                model_metadata={"outcome": job.outcome, "family": job.family, "config_id": job.config_id, "model_id": job.artifact_id, "refit_id": "pilot_deployment"},
                background_id=background_id, approximation_seed=approximation_seed,
                n_orderings=int(PROFILE["permutation_orderings"]),
            ))
            runtime_rows.append({
                "outcome": job.outcome, "family": job.family, "background_id": background_id,
                "approximation_seed": approximation_seed, "cases": len(pilot_cases),
                "seconds": time.perf_counter() - started,
            })
attributions = pd.concat(attribution_parts, ignore_index=True)
predictions_2008 = pd.concat(prediction_parts, ignore_index=True)
diagnostics = approximation_repeat_diagnostics(attributions)

In [ ]:
# Diagnose the mixed-type conditional donor model, then create development-only
# operational explanation losses for the cross-fitted deployment-time meta-score.
sampler = GowerKNNDonorSampler.fit(
    development,
    feature_columns,
    k=int(statistical["faithfulness"]["conditional_sampler_primary_k"]),
)
sampler_diagnostics = sampler.diagnostics(pilot_cases.head(min(100, len(pilot_cases))))

primary_job = (
    jobs[jobs["outcome"].eq("F36")]
    .sort_values(["platt_oof_log_loss", "family", "config_id"])
    .iloc[0]
)
primary_model = joblib.load(primary_job["calibrated_model_path"])
first_background_id = active_background_ids[0]
conditional = explain_dataset(
    primary_model,
    pilot_cases.head(min(40, len(pilot_cases))),
    backgrounds[first_background_id],
    groups,
    feature_columns,
    model_metadata={
        "outcome": "F36",
        "family": primary_job["family"],
        "config_id": primary_job["config_id"],
        "model_id": primary_job["artifact_id"],
        "refit_id": "pilot_deployment",
    },
    background_id=first_background_id,
    approximation_seed=approximation_seeds[0],
    n_orderings=max(4, int(PROFILE["permutation_orderings"])),
    method="conditional_imputation",
    conditional_sampler=sampler,
)

primary_attr = attributions[
    attributions["outcome"].eq("F36")
    & attributions["family"].eq(primary_job["family"])
    & attributions["config_id"].eq(primary_job["config_id"])
].copy()
primary_mean_attr = (
    primary_attr.groupby(["case_id", "feature_group"], as_index=False)
    .agg(phi_log_odds=("phi_log_odds", "mean"))
)
f36_panel = attributions[attributions["outcome"].eq("F36")].copy()
f36_panel["panel_id"] = (
    f36_panel["family"].astype(str) + "__" + f36_panel["config_id"].astype(str)
    + "__" + f36_panel["background_id"].astype(str)
    + "__s" + f36_panel["approximation_seed"].astype(str)
)
pilot_instability = case_reference_panel_loss(
    primary_mean_attr,
    f36_panel,
    group_order=sorted(groups),
)
construct_attr = attributions[
    attributions["family"].eq(primary_job["family"])
    & attributions["config_id"].eq(primary_job["config_id"])
    & attributions["outcome"].isin(CFG["outcomes"]["confirmatory"])
].copy()
pilot_construct = case_construct_fragility_loss(
    construct_attr,
    group_order=sorted(groups),
)

pilot_faithfulness_curves, pilot_faithfulness = faithfulness_audit(
    primary_model,
    pilot_cases,
    primary_mean_attr,
    groups,
    sampler,
    feature_columns,
    steps=tuple(statistical["faithfulness"]["deletion_steps"]),
    random_repetitions=int(PROFILE["esrc_faithfulness_random_repetitions"]),
    seed=int(CFG["execution"]["random_seed"]) + 90,
)
positive_faithfulness_signal = pilot_faithfulness.loc[
    pilot_faithfulness["top_minus_random_aopc"].gt(0), "top_minus_random_aopc"
]
faithfulness_scale = max(
    float(positive_faithfulness_signal.quantile(0.75)) if not positive_faithfulness_signal.empty else 0.0,
    float(CFG["selective_risk"]["faithfulness_scale_floor"]),
)
pilot_faithfulness["L_F"] = normalized_faithfulness_loss(
    pilot_faithfulness["top_aopc"],
    pilot_faithfulness["random_aopc_mean"],
    faithfulness_scale,
)
pilot_losses = (
    pilot_cases[["case_id"]]
    .merge(pilot_instability[["case_id", "L_S"]], on="case_id", how="left", validate="one_to_one")
    .merge(pilot_construct[["case_id", "L_C"]], on="case_id", how="left", validate="one_to_one")
    .merge(pilot_faithfulness[["case_id", "L_F"]], on="case_id", how="left", validate="one_to_one")
)
pilot_losses[["L_S", "L_C", "L_F"]] = pilot_losses[["L_S", "L_C", "L_F"]].fillna(1.0).clip(0, 1)
pilot_losses["meta_target_LS_LF"] = pilot_losses[["L_S", "L_F"]].mean(axis=1)

In [ ]:
# The meta-score predicts actual development-only instability/faithfulness loss,
# not merely explainer Monte Carlo variance. Its performance is evaluated by
# internal cross-fitting; the final model is then frozen for 2009/2010 scoring.
meta_data = pilot_cases.merge(pilot_losses, on="case_id", how="inner", validate="one_to_one")
feature_types = infer_feature_types(meta_data, feature_columns)
preprocessor = build_preprocessor(feature_types, scale_numeric=False)
meta_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=10,
        random_state=int(CFG["execution"]["random_seed"]),
        n_jobs=-1,
    )),
])
folds = min(5, max(2, len(meta_data) // 40))
meta_cv = KFold(n_splits=folds, shuffle=True, random_state=int(CFG["execution"]["random_seed"]) + 91)
meta_oof_prediction = cross_val_predict(
    meta_model,
    meta_data[feature_columns],
    meta_data["meta_target_LS_LF"],
    cv=meta_cv,
    n_jobs=1,
    method="predict",
)
meta_evaluation = meta_data[["case_id", "L_S", "L_C", "L_F", "meta_target_LS_LF"]].copy()
meta_evaluation["oof_predicted_explanation_loss"] = np.clip(meta_oof_prediction, 0, 1)
meta_evaluation["absolute_error"] = (
    meta_evaluation["oof_predicted_explanation_loss"] - meta_evaluation["meta_target_LS_LF"]
).abs()
meta_model.fit(meta_data[feature_columns], meta_data["meta_target_LS_LF"])
meta_path = P.models / "predicted_explanation_loss_meta_model.joblib"
joblib.dump(meta_model, meta_path)

density_types = infer_feature_types(development, feature_columns)
density_preprocessor = build_preprocessor(density_types, scale_numeric=True)
density_model = Pipeline([
    ("preprocess", density_preprocessor),
    ("model", IsolationForest(
        n_estimators=500,
        contamination="auto",
        random_state=int(CFG["execution"]["random_seed"]),
        n_jobs=-1,
    )),
])
density_model.fit(development[feature_columns])
density_path = P.models / "development_density_model.joblib"
joblib.dump(density_model, density_path)

In [ ]:
primary_prediction = predictions_2008[
    predictions_2008["outcome"].eq("F36")
    & predictions_2008["family"].eq(primary_job["family"])
    & predictions_2008["config_id"].eq(primary_job["config_id"])
].set_index("case_id")["probability"]
primary_prob = primary_prediction.reindex(calibration_2008["case_id"]).to_numpy(dtype=float)
if np.isnan(primary_prob).any():
    raise RuntimeError("Frozen F36 deployment probabilities are incomplete on the 2008 block")
meta_score = np.clip(meta_model.predict(calibration_2008[feature_columns]), 0, 1)
density_score = -density_model.decision_function(calibration_2008[feature_columns])
gate_scores = standard_gate_scores(
    primary_prob,
    density_score=density_score,
    predicted_explanation_loss=meta_score,
)
gate_scores.insert(0, "case_id", calibration_2008["case_id"].to_numpy())

attr_path = write_table(attributions, P.attributions / "development_pilot_attributions.parquet")
conditional_path = write_table(conditional, P.attributions / "development_pilot_conditional_imputation.parquet")
diagnostics_path = write_table(diagnostics, P.audits / "09_approximation_repeat_diagnostics.csv")
runtime_path = write_table(pd.DataFrame(runtime_rows), P.audits / "09_explanation_runtime.csv")
sampler_path = write_json(sampler_diagnostics, P.audits / "09_conditional_sampler_diagnostics.json")
gate_path = write_table(gate_scores, P.selective / "development_gate_scores_2008.parquet")
prediction_path = write_table(predictions_2008, P.predictions / "development_pilot_predictions_2008.parquet")
pilot_loss_path = write_table(pilot_losses, P.selective / "development_explanation_losses_2008.parquet")
faithfulness_curve_path = write_table(
    pilot_faithfulness_curves,
    P.controls / "development_faithfulness_curves_2008.parquet",
)
faithfulness_path = write_table(
    pilot_faithfulness,
    P.controls / "development_faithfulness_summary_2008.parquet",
)
scale_path = write_json({
    "scale_name": "s_F",
    "value": faithfulness_scale,
    "source_block": "probability_calibration_2008",
    "definition": "max(75th percentile positive top-minus-random conditional AOPC, frozen floor)",
    "floor": float(CFG["selective_risk"]["faithfulness_scale_floor"]),
    "primary_family": str(primary_job["family"]),
    "primary_config_id": str(primary_job["config_id"]),
}, P.protocol / "development_faithfulness_scale.json")
meta_oof_path = write_table(meta_evaluation, P.audits / "09_explanation_loss_meta_oof.csv")
CTX.recorder.complete([
    attr_path, conditional_path, diagnostics_path, runtime_path, sampler_path,
    gate_path, prediction_path, pilot_loss_path, faithfulness_curve_path,
    faithfulness_path, scale_path, meta_oof_path, meta_path, density_path,
], extra={
    "primary_gate_family": str(primary_job["family"]),
    "primary_gate_config_id": str(primary_job["config_id"]),
    "faithfulness_scale": faithfulness_scale,
    "meta_oof_mae": float(meta_evaluation["absolute_error"].mean()),
})
print(sampler_diagnostics)
print({
    "faithfulness_scale": faithfulness_scale,
    "meta_oof_mae": float(meta_evaluation["absolute_error"].mean()),
    "meta_oof_correlation": float(meta_evaluation[["meta_target_LS_LF", "oof_predicted_explanation_loss"]].corr().iloc[0, 1]),
})